# 07 Extracts for the team (data/exports/)

`python -m etl.exports`. For Wanru (MinHash LSH on comments and titles), Zhenni (k-anonymity and Laplace noise) and the charts. Schemas and regeneration: `docs/final/HANDOFF_TEAM.md`. Human editors appear only as pseudonyms.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])
m = json.loads((REPO / 'data/exports/manifest.json').read_text())
print(m['lake_versions'])
pd.DataFrame(m['files']).T[['rows', 'bytes']]

{'fact_edit': 3, 'dim_page': 3, 'fact_revert_label': 1, 'mart_el_curve': 2, 'mart_el_by_burst': 2, 'silver/history_revisions': 1, 'silver/live_revisions': 3, 'silver/live_ext_revisions': 1}


,rows,bytes
lsh_edits.parquet,261166,17407748
lsh_edits.csv,261166,44755394
mart_el_curve.csv,117,9000
mart_el_by_burst.csv,162,12097
group_sizes_by_kind_hour.csv,1875,86391
page_revert_summary.parquet,1559483,39435245
page_revert_summary_top.csv,88503,4572188
hypotheses_h1.csv,4,784
hypotheses_h3.csv,8,1309
hypotheses_h4.json,1,893


In [2]:
lsh = pd.read_parquet(REPO / 'data/exports/lsh_edits.parquet')
lsh.groupby(['cohort_id', 'label']).size()

cohort_id  label
history    0         17646
           1         17646
live_ext   0        112937
           1        112937
dtype: int64

In [3]:
g = pd.read_csv(REPO / 'data/exports/group_sizes_by_kind_hour.csv')
g.describe()

,hour_utc,revisions,reverted_4d,distinct_editors,distinct_page_versions
count,1875.000000,1875.000000,1875.000000,1875.000000,1875.000000
mean,11.513067,1585.184000,79.147200,387.049600,1243.010133
std,6.927141,1315.859756,62.387604,379.435762,1017.976694
min,0.000000,1.000000,0.000000,1.000000,1.000000
25%,6.000000,448.000000,4.000000,12.000000,337.000000
50%,12.000000,864.000000,89.000000,265.000000,863.000000
75%,18.000000,2726.500000,123.500000,722.500000,2072.000000
max,23.000000,5059.000000,446.000000,1192.000000,4754.000000
